<center><h1>Last_First_HW7</h1></center>

Name: Tony Chen
<br>
Github Username: tchenusc
<br>
USC ID: 9747866232

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from imblearn.over_sampling import SMOTE

### (a) Download the Anuran Calls (MFCCs) Data Set

In [2]:
df = pd.read_csv('../data/Anuran Calls (MFCCs)/Frogs_MFCCs.csv')
labelList = ['Family', 'Genus', 'Species']
featureList = df.columns[:22]

trainDf, testDf = train_test_split(df, train_size=0.7, random_state=42)
xTrainRaw = trainDf[featureList]
xTestRaw = testDf[featureList]
yTest = testDf[labelList]

### (b) Train a classifier for each label

#### (i) Research

The Exact Match Ratio (also called Subset Accuracy) is the most strict metric, indicating the percentage of samples that have all their labels classified correctly.

The Hamming score is a Classification metric featuring a fraction of the predictions that a model got right

#### (ii) Train a SVM for each of the labels

In [ ]:
paramGrid = {'estimator__C': [1, 10, 100, 1000], 'estimator__gamma': [0.01, 0.1, 1, 10]}

predictionsRaw = pd.DataFrame()
for label in labelList:
    gridSearch = GridSearchCV(OneVsRestClassifier(SVC(kernel='rbf')), paramGrid, cv=10, n_jobs=-1)
    gridSearch.fit(xTrainRaw, trainDf[label])
    
    print(label, gridSearch.best_params_)
    predictionsRaw[label] = gridSearch.predict(xTestRaw)

correctRaw = predictionsRaw.values == yTest.values
print('Exact match:', correctRaw.all(axis=1).mean())
print('Hamming score:', correctRaw.mean())

Family {'estimator__C': 100, 'estimator__gamma': 1}


Genus {'estimator__C': 10, 'estimator__gamma': 1}


Species {'estimator__C': 10, 'estimator__gamma': 1}


Exact match: 0.9874942102825383
Hamming score: 0.9922803767176162


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [4]:
scaler = StandardScaler()
xTrainStandardized = scaler.fit_transform(xTrainRaw)
xTestStandardized = scaler.transform(xTestRaw)

paramGridL1 = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

predictionsL1 = pd.DataFrame()
for label in labelList:
    gridSearch = GridSearchCV(LinearSVC(penalty='l1', dual=False, max_iter=10000), paramGridL1, cv=10, n_jobs=-1)
    gridSearch.fit(xTrainStandardized, trainDf[label])
    print(label, gridSearch.best_params_)
    predictionsL1[label] = gridSearch.predict(xTestStandardized)

correctL1 = predictionsL1.values == yTest.values
print('Exact match:', correctL1.all(axis=1).mean())
print('Hamming score:', correctL1.mean())

Family {'C': 1}


/Users/starfengsmac/Desktop/DSCI 552/HW1/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/starfengsmac/Desktop/DSCI 552/HW1/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Genus {'C': 10}


/Users/starfengsmac/Desktop/DSCI 552/HW1/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/starfengsmac/Desktop/DSCI 552/HW1/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Species {'C': 10}
Exact match: 0.9129226493747105
Hamming score: 0.9431835726416551


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [5]:
predictionsSmote = pd.DataFrame()
for label in labelList:
    xTrainSmote, yTrainSmote = SMOTE(random_state=42).fit_resample(xTrainStandardized, trainDf[label])
    gridSearch = GridSearchCV(LinearSVC(penalty='l1', dual=False, max_iter=10000), paramGridL1, cv=10, n_jobs=-1)
    gridSearch.fit(xTrainSmote, yTrainSmote)
    print(label, gridSearch.best_params_)
    predictionsSmote[label] = gridSearch.predict(xTestStandardized)

correctSmote = predictionsSmote.values == yTest.values
print('Exact match:', correctSmote.all(axis=1).mean())
print('Hamming score:', correctSmote.mean())

Family {'C': 100}


Genus {'C': 100}


Species {'C': 100}
Exact match: 0.8531727651690597
Hamming score: 0.9229581596418095


The Gaussian SVM scored the highest with an exact match of 0.987 and a Hamming score of 0.992. The L1 penalized SVMs scored the second best with a 0.913 exact match score and a Hamming score of 0.943. Lastly was SMOTE, with an exact match score of 0.853 and a Hamming score of 0.923.

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [6]:
silhouetteScoreList = []
for k in range(2, 51):
    clusterLabels = KMeans(n_clusters=k, random_state=0).fit_predict(df[featureList])
    silhouetteScoreList.append(silhouette_score(df[featureList], clusterLabels))

bestK = silhouetteScoreList.index(max(silhouetteScoreList)) + 2
print('Best k:', bestK)

Best k: 4


### (b) Determine which family is the majority

In [ ]:
clusterLabels = KMeans(n_clusters=bestK, random_state=0).fit_predict(df[featureList])

majorityDf = pd.DataFrame(index=df.index, columns=labelList)
for cluster in range(bestK):
    inCluster = clusterLabels == cluster
    print('Cluster', cluster)
    for label in labelList:
        majorityLabel = df.loc[inCluster, label].value_counts().idxmax()
        majorityDf.loc[inCluster, label] = majorityLabel
        
        print(label, majorityLabel)

Cluster 0
Family Leptodactylidae
Genus Adenomera
Species AdenomeraHylaedactylus
Cluster 1
Family Hylidae
Genus Hypsiboas
Species HypsiboasCordobae
Cluster 2
Family Hylidae
Genus Hypsiboas
Species HypsiboasCinerascens
Cluster 3
Family Leptodactylidae
Genus Adenomera
Species AdenomeraAndre


### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [ ]:
hammingDistanceList = []
hammingScoreList = []
hammingLossList = []

for run in range(50):
    bestSilhouette = -1
    for k in range(2, 51):
        clusterLabels = KMeans(n_clusters=k, random_state=run).fit_predict(df[featureList])
        silhouette = silhouette_score(df[featureList], clusterLabels)

        if silhouette > bestSilhouette:
            bestSilhouette = silhouette
            bestClusterLabels = clusterLabels
            bestK = k

    majorityDf = pd.DataFrame(index=df.index, columns=labelList)
    for cluster in range(bestK):
        inCluster = bestClusterLabels == cluster
        
        for label in labelList:
            majorityDf.loc[inCluster, label] = df.loc[inCluster, label].value_counts().idxmax()

    hammingDistance = (majorityDf.values != df[labelList].values).sum(axis=1).mean()
    hammingDistanceList.append(hammingDistance)
    hammingScoreList.append(1 - hammingDistance / 3)
    hammingLossList.append(hammingDistance / 3)

print('Hamming distance:', np.mean(hammingDistanceList), np.std(hammingDistanceList))
print('Hamming score:', np.mean(hammingScoreList), np.std(hammingScoreList))
print('Hamming loss:', np.mean(hammingLossList), np.std(hammingLossList))

Hamming distance: 0.7186685198054203 0.12959433572039117
Hamming score: 0.7604438267315266 0.04319811190679705
Hamming loss: 0.23955617326847353 0.04319811190679705


## 3. ISLR 12.6.2

![](ISLR_12_6_2.png)